In [2]:
pip install pytest-playwright

In [3]:
playwright install chromium

SyntaxError: invalid syntax (1470437723.py, line 1)

In [1]:
import re
import pandas as pd
from playwright.sync_api import sync_playwright

BASE = "https://www.eleicoes.mai.gov.pt/legislativas2024/resultados/territorio-nacional?local={local}"

# Porto 下辖 18 个 concelho：LOCAL-130100 ... LOCAL-131800（步长 100）
LOCALS = [f"LOCAL-{x}" for x in range(130100, 131801, 100)]

OUT_XLSX = "porto_concelhos_parties_votes_share.xlsx"

def clean_int(s: str) -> int:
    # 处理 "12.345" 这种葡语千分位
    s = re.sub(r"[^\d]", "", s)
    return int(s) if s else 0

def clean_pct(s: str) -> float:
    # 处理 "12,34%" -> 12.34
    s = s.strip().replace("%", "").replace(".", "").replace(",", ".")
    try:
        return float(s)
    except:
        return float("nan")

def scrape_one(page, local_code: str):
    url = BASE.format(local=local_code)
    page.goto(url, wait_until="networkidle", timeout=120_000)

    # 站点是前端渲染：等“政党结果表格/列表”出现
    # 下面这段 selector 逻辑做了“宽松匹配”，避免因页面小改动就挂掉
    page.wait_for_timeout(2000)

    # 尝试从页面中抓 Concelho 名称（标题/面包屑）
    concelho = None
    for sel in [
        "h1", "h2",
        "[class*='title']",
        "[class*='breadcrumb']"
    ]:
        el = page.query_selector(sel)
        if el:
            t = (el.inner_text() or "").strip()
            if t and len(t) <= 80:
                concelho = t
                break
    concelho = concelho or local_code

    # 核心：抓“政党-百分比-票数”
    #
    # 由于该站点常见呈现方式是每个党一行，行内同时出现：
    #   - 党名（例如 PS / CH / IL / BE / CDU / L / PAN / ...）
    #   - 百分比（含 %）
    #   - 票数（含千分位点）
    #
    # 我们直接在 DOM 里找所有包含 “%” 的条目，然后在其附近提取党名与票数。
    #
    # 注意：如果你发现输出缺列/错位，告诉我你页面里“结果列表”的 DOM 结构（F12复制一小段），我可以把 selector 精准化。
    items = page.query_selector_all("text=%")
    rows = []

    for it in items:
        # 往上找一个“行容器”
        container = it
        for _ in range(6):
            parent = container.query_selector("xpath=..")
            if not parent:
                break
            container = parent

        block = (container.inner_text() or "").strip()
        if not block:
            continue

        # 粗略过滤：必须同时有百分比和“Votos/票数格式”
        if "%" not in block:
            continue

        # 常见 block 例子（示意）：
        # "PPD/PSD.CDS-PP.PPM 44,08% 5.919 Votos"
        # "PS 27,69% 3.712 Votos"
        #
        # 提取：党名、百分比、票数
        m_pct = re.search(r"(\d{1,2}(?:[.,]\d{1,2})?)\s*%", block)
        m_votes = re.search(r"(\d{1,3}(?:\.\d{3})*|\d+)\s*(?:Votos|votos)", block)
        if not (m_pct and m_votes):
            continue

        pct = clean_pct(m_pct.group(0))
        votes = clean_int(m_votes.group(1))

        # 党名：取百分比前的文本，去掉多余换行
        party_raw = block[:m_pct.start()].strip()
        party_raw = re.sub(r"\s+", " ", party_raw)

        # 再过滤一下明显不是党名的内容
        if len(party_raw) == 0 or len(party_raw) > 60:
            continue

        rows.append({
            "district": "Porto",
            "local_code": local_code,
            "concelho": concelho,
            "party": party_raw,
            "votes": votes,
            "share_pct": pct
        })

    # 去重（同一 block 可能被多次命中）
    if rows:
        df = pd.DataFrame(rows).drop_duplicates(subset=["local_code", "party", "votes", "share_pct"])
        return df
    else:
        return pd.DataFrame(columns=["district","local_code","concelho","party","votes","share_pct"])

def main():
    all_df = []
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        for local_code in LOCALS:
            df = scrape_one(page, local_code)
            all_df.append(df)
            print(f"{local_code}: {len(df)} rows")
        browser.close()

    out = pd.concat(all_df, ignore_index=True)
    # 排序更好看
    out = out.sort_values(["local_code", "votes"], ascending=[True, False])

    out.to_excel(OUT_XLSX, index=False)
    print(f"\nSaved: {OUT_XLSX}\nTotal rows: {len(out)}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'playwright'